In [9]:
%pip install peft evaluate
%pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 27.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
torch.cuda.is_available()

True

In [5]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
except:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print(e)

In [17]:
labels = {
    0: "AG",
    1: "BE",
    2: "BS",
    3: "GR",
    4: "LU",
    5: "SG",
    6: "VS",
    7: "ZH",
}

id2label = labels
label2id = {v: k for k, v in labels.items()}
num_labels = len(labels)

In [18]:
from transformers import AutoModelForAudioClassification, AutoConfig, AutoFeatureExtractor, Wav2Vec2Processor
import torch

model_id = "facebook/wav2vec2-xls-r-300m"

feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)

config = AutoConfig.from_pretrained(
    model_id,
    num_labels=8,
    label2id=label2id,
    id2label=id2label,
)

model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    config=config
)

model.freeze_feature_encoder()

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
from peft import PeftModel, LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["projector", "classifier"],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 3,410,184 || all params: 319,113,360 || trainable%: 1.0686


In [11]:
import datasets
from datasets import load_dataset, load_from_disk, Audio, interleave_datasets, concatenate_datasets, Value

# Training data: Combine SwissDial and ArchiMob and balance by region
ds_swissdial = load_dataset("RobChio/swiss-dial-preprocessed", split="train")
ds_archimob_train = load_dataset("RobChio/archimob-preprocessed", split="train")

ds_swissdial = ds_swissdial.cast_column("dialect_code", Value(dtype="int64")) # fix type mismatch

ds_swissdial.set_format(type="torch", columns=["audio", "dialect_code"])
ds_archimob_train.set_format(type="torch", columns=["audio", "dialect_code"])

combined_train = concatenate_datasets([ds_swissdial, ds_archimob_train])
train_dataset = combined_train.shuffle(seed=42)
#train_dataset = balance_dataset(combined_train, label_col="dialect_code", seed=42)
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))

# Validation data from ArchiMob
ds_archimob_val = load_dataset("RobChio/archimob-preprocessed", split="validation", streaming=False)
val_dataset = ds_archimob_val.shuffle(seed=42)
val_dataset.set_format(type="torch", columns=["audio", "dialect_code"])
val_dataset = val_dataset.cast_column("audio", Audio(sampling_rate=16000))

data-00000-of-00008.arrow:   0%|          | 0.00/492M [00:00<?, ?B/s]

data-00001-of-00008.arrow:   0%|          | 0.00/540M [00:00<?, ?B/s]

data-00002-of-00008.arrow:   0%|          | 0.00/505M [00:00<?, ?B/s]

data-00003-of-00008.arrow:   0%|          | 0.00/534M [00:00<?, ?B/s]

data-00004-of-00008.arrow:   0%|          | 0.00/532M [00:00<?, ?B/s]

...

validation/data-00001-of-00003.arrow:   0%|          | 0.00/372M [00:00<?, ?B/s]

validation/data-00002-of-00003.arrow:   0%|          | 0.00/312M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Casting the dataset:   0%|          | 0/30921 [00:00<?, ? examples/s]

In [16]:
def preprocess_function(batch):
    audios = [x["array"] for x in batch["audio"]]
    inputs = feature_extractor(
        audios,
        sampling_rate=16000,
        max_length=160000,  # Truncate at 10 seconds (16,000 Hz * 10s)
        truncation=True,
    )
    inputs["label"] = batch["dialect_code"]
    return inputs

# Apply mapping (audio is loaded on the fly)
train_dataset_preprocessed = train_dataset.map(
    preprocess_function,
    remove_columns=["audio"],
    batched=True,
    batch_size=64,
    num_proc=4,
)

val_dataset_preprocessed = val_dataset.map(
    preprocess_function,
    remove_columns=["audio"],
    batched=True,
    batch_size=64,
    num_proc=4,
)

Map (num_proc=4):   0%|          | 0/70769 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/10611 [00:00<?, ? examples/s]

In [14]:
from dataclasses import dataclass
from typing import Dict, List, Union

@dataclass
class ClassificationDataCollator:
    feature_extractor: AutoFeatureExtractor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_values = [{"input_values": feature["input_values"]} for feature in features]
        labels = [feature["label"] for feature in features]

        batch = self.feature_extractor.pad(
            input_values,
            padding=self.padding,
            return_tensors="pt",
        )
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

data_collator = ClassificationDataCollator(feature_extractor=feature_extractor)

In [20]:
from torch.nn.utils.rnn import pad_sequence

class FastClassificationDataCollator:
    def __call__(self, features):
        tensors = [
            torch.as_tensor(f["input_values"], dtype=torch.float32) 
            for f in features
        ]
        # Dynamic CPU padding to the longest sequence in the current batch
        input_values = pad_sequence(tensors, batch_first=True, padding_value=0.0)
        labels = torch.tensor([f["label"] for f in features], dtype=torch.long)
        return {"input_values": input_values, "labels": labels}

data_collator = FastClassificationDataCollator()

In [22]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np
import torch.nn.functional as F

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    
    macro_f1 = f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    acc = accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    return {"f1": macro_f1, "accuracy": acc}

training_args = TrainingArguments(
    output_dir="./swissgerman-dialect-classifier-xls-r",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    dataloader_num_workers=4, # multiple workers with streaming causes data duplication
    dataloader_pin_memory=True,
    dataloader_persistent_workers=True,
    dataloader_prefetch_factor=2,
    torch_compile=False, #True,
    dataloader_drop_last=False,
    push_to_hub=False,
    hub_model_id="RobChio/swissgerman-dialect-classifier-xls-r",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_preprocessed,
    eval_dataset=val_dataset_preprocessed,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
train_result = trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


In [ ]:
#trainer.push_to_hub(commit_message="Finished training")

In [ ]:
# from google.colab import runtime
# runtime.unassign()